In [10]:
%pip install optuna rank_bm25 sentence-transformers scikit-learn river matplotlib pyyaml tqdm kaggle kagglehub


   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import shutil

# 1. Establish paths securely for Windows
home_dir = os.path.expanduser("~")
target_dir = os.path.join(home_dir, ".kaggle")
target_file_path = os.path.join(target_dir, "kaggle.json")

# Create the C:\Users\Username\.kaggle directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# 2. Check both potential workspace locations for your credential file
possible_sources = [
    os.path.join(".kaggle", "kaggle.json"),  # Checked first based on your explorer
    "kaggle.json"                            # Root directory fallback
]

file_moved = False
for source in possible_sources:
    if os.path.exists(source):
        shutil.copy(source, target_file_path)
        print(f"✅ Successfully initialized Kaggle credentials from: {source}")
        file_moved = True
        break

if not file_moved:
    print("❌ Critical: 'kaggle.json' could not be located in your workspace.")
    print("   Please ensure it is placed in your project folder or inside the local .kaggle subfolder.")

✅ Successfully initialized Kaggle credentials from: .kaggle\kaggle.json


In [12]:

# =============================================================================
# Track A: Supervised Auto-tuned kNN Retriever (Optuna)g
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import time, json, yaml, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from rank_bm25 import BM25Okapi
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

from river import linear_model, preprocessing, metrics, drift
from river.stream import iter_array

warnings.filterwarnings("ignore")
np.random.seed(42)

DEVICE      = "cpu"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
N_TRIALS    = 40          # Optuna trials (raise to 80+ for final run)
TOP_K_EVAL  = 5           # NDCG@5 / Recall@5
DATA_SUBSET = 500         # papers to use (max out at what the dataset has)

print("Imports ready & done")

Imports ready & done


In [14]:
# ── Cell 2: Load Kaggle arXiv Dataset ────────────────────────────────────────

import kagglehub
import os

print("Downloading arXiv dataset from Kaggle …")
dataset_path = kagglehub.dataset_download(
    "sumitm004/arxiv-scientific-research-papers-dataset"
)
print(f"Downloaded to: {dataset_path}")

# ── Auto-find the CSV/parquet file inside the downloaded folder ───────────────
all_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"Files found: {all_files}")

# Pick the first CSV or parquet file found
data_file = next(
    (f for f in all_files if f.endswith((".csv", ".parquet", ".tsv"))), None
)
assert data_file, f" No CSV/parquet file found. Files present: {all_files}"
print(f" Loading: {data_file}")

raw_df = pd.read_csv(data_file) if data_file.endswith((".csv", ".tsv")) \
         else pd.read_parquet(data_file)
print(f"Raw dataset shape : {raw_df.shape}")
print(f"Columns           : {list(raw_df.columns)}")

# ── Normalise column names (lowercase + strip whitespace) ────────────────────
raw_df.columns = raw_df.columns.str.strip().str.lower().str.replace(" ", "_")

# ── Identify the abstract/text column ────────────────────────────────────────
# Common names in arXiv Kaggle datasets — adjust if yours differs
TEXT_COL     = next((c for c in raw_df.columns if c in
                     ["abstract", "summary", "description", "text"]), None)
TITLE_COL    = next((c for c in raw_df.columns if "title"    in c), None)
CAT_COL      = next((c for c in raw_df.columns if "categor"  in c), None)
AUTHOR_COL   = next((c for c in raw_df.columns if "author"   in c), None)
ID_COL       = next((c for c in raw_df.columns if c in
                     ["id", "arxiv_id", "paper_id"]), None)

print(f"\n Detected columns → text: '{TEXT_COL}' | title: '{TITLE_COL}' "
      f"| category: '{CAT_COL}' | id: '{ID_COL}'")

assert TEXT_COL, " Could not find an abstract/text column — check raw_df.columns above"

# ── Clean & filter ────────────────────────────────────────────────────────────
df = raw_df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].str.strip()
df = df[df[TEXT_COL].str.len() > 80]           # drop very short abstracts

# Filter to AI/ML papers if category column exists
if CAT_COL:
    ai_mask = df[CAT_COL].str.contains(
        "cs.AI|cs.CL|cs.LG|cs.CV|stat.ML", na=False, case=False)
    df_ai = df[ai_mask]
    df = df_ai if len(df_ai) >= 100 else df    # fall back to full set if too small
    print(f"AI/ML papers found: {len(df_ai)} | Using: {len(df)}")

# Sample to DATA_SUBSET for speed
df = df.sample(min(DATA_SUBSET, len(df)), random_state=42).reset_index(drop=True)

# ── Build the lists the rest of the notebook expects ─────────────────────────
texts   = df[TEXT_COL].tolist()
doc_ids = (df[ID_COL].astype(str).tolist()
           if ID_COL else [f"doc_{i:04d}" for i in range(len(df))])
labels  = (df[CAT_COL].astype(str).tolist()
           if CAT_COL else ["unknown"] * len(df))

# ── Gold Q/A set: titles as queries, abstracts as corpus ─────────────────────
# Using the full abstract as both query AND document is trivially easy for
# dense embeddings (cosine sim = 1.0 always → NDCG = 1.0 always → nothing
# for Optuna to optimise).
#
# Instead: query = title (short, keyword-like)
#          corpus = abstract (longer, semantic)
# This mirrors real usage and is hard enough that hyperparams actually matter.

assert TITLE_COL, " No title column found — check your dataset columns"

GOLD_N = min(100, len(df) // 4)

# Build gold pairs: (title → doc_id) for first GOLD_N rows
gold_queries = df[TITLE_COL].iloc[:GOLD_N].fillna("").tolist()
gold_doc_ids = doc_ids[:GOLD_N]

# Corpus = ALL abstracts (gold docs included so retriever can find them)
train_texts  = texts        # abstracts
train_ids    = doc_ids
train_labels = labels

print(f"Gold queries (titles) : {GOLD_N}")
print(f"Corpus (abstracts)    : {len(train_texts)}")
print(f"\nExample query : '{gold_queries[0]}'")
print(f"Expected doc  : '{gold_doc_ids[0]}'")

# ── Quick sanity-check preview ────────────────────────────────────────────────
print(f"\n Corpus ready: {len(train_texts)} docs | Gold queries: {GOLD_N}")
print(f"\nSample abstract:\n  {train_texts[0][:200]} …")
if TITLE_COL:
    print(f"Sample title    :\n  {df[TITLE_COL].iloc[GOLD_N]}")

Downloaded to: C:\Users\Baraa\.cache\kagglehub\datasets\sumitm004\arxiv-scientific-research-papers-dataset\versions\2
Files found: ['C:\\Users\\Baraa\\.cache\\kagglehub\\datasets\\sumitm004\\arxiv-scientific-research-papers-dataset\\versions\\2\\arXiv_scientific dataset.csv']
 Loading: C:\Users\Baraa\.cache\kagglehub\datasets\sumitm004\arxiv-scientific-research-papers-dataset\versions\2\arXiv_scientific dataset.csv
Raw dataset shape : (136238, 10)
Columns           : ['id', 'title', 'category', 'category_code', 'published_date', 'updated_date', 'authors', 'first_author', 'summary', 'summary_word_count']

 Detected columns → text: 'summary' | title: 'title' | category: 'category' | id: 'id'
AI/ML papers found: 0 | Using: 136206
Gold queries (titles) : 100
Corpus (abstracts)    : 500

Example query : 'Multi-hop assortativities for networks classification'
Expected doc  : 'abs-1809.06253v2'

 Corpus ready: 500 docs | Gold queries: 100

Sample abstract:
  Several social, medical, engineeri

In [15]:
#  ── Cell 3: Embeddings ────────────────────────────────────────────────────────
print("Encoding dense embeddings …")
encoder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

corpus_emb = encoder.encode(train_texts, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)
# Gold queries are now titles (short), corpus is abstracts (long) — intentionally asymmetric
query_emb  = encoder.encode(gold_queries, batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)

print(f"Embedding shape: {corpus_emb.shape}")


Encoding dense embeddings …


Batches: 100%|██████████| 2/2 [00:00<00:00,  7.92it/s]

Embedding shape: (500, 384)


In [16]:
#  ── Cell 4: BM25 Index ───────────────────────────────────────────────────────
tokenized = [t.lower().split() for t in train_texts]
bm25      = BM25Okapi(tokenized)

def bm25_scores(query: str) -> np.ndarray:
    """Return normalised BM25 scores for all corpus docs."""
    raw_s = np.array(bm25.get_scores(query.lower().split()))
    mx = raw_s.max()
    return raw_s / mx if mx > 0 else raw_s

print("BM25 index built")

BM25 index built


In [17]:
# ── Cell 5: Hybrid Retriever Class ────────────────────────────────────────────
class HybridRetriever:
    """
    BM25 + Dense hybrid retriever with configurable:
      alpha       — weight of BM25 (1-alpha = dense weight)
      svd_dim     — TruncatedSVD output dim (None = no reduction)
      norm        — 'l2' | 'none'
      metric      — 'cosine' | 'euclidean' | 'dot' (via NearestNeighbors)
      k           — number of neighbours to return
    """

    def __init__(self, alpha=0.5, svd_dim=None, norm="l2",
                 metric="cosine", k=5):
        self.alpha = alpha
        self.svd_dim = svd_dim
        self.norm = norm
        self.metric = metric
        self.k = k
        self._svd = None
        self._index = None
        self._dense = None

    def fit(self, dense_matrix: np.ndarray):
        d = dense_matrix.copy()
        if self.svd_dim and self.svd_dim < d.shape[1]:
            self._svd = TruncatedSVD(n_components=self.svd_dim, random_state=42)
            d = self._svd.fit_transform(d)
        if self.norm == "l2":
            d = normalize(d, norm="l2")
        self._dense = d
        nn_metric = "cosine" if self.metric in ("cosine", "dot") else "euclidean"
        self._index = NearestNeighbors(n_neighbors=self.k,
                                       metric=nn_metric, algorithm="brute")
        self._index.fit(d)
        return self

    def query(self, q_text: str, q_emb: np.ndarray) -> list[str]:
        # Dense side
        qd = q_emb.copy().reshape(1, -1)
        if self._svd:
            qd = self._svd.transform(qd)
        if self.norm == "l2":
            qd = normalize(qd, norm="l2")

        dense_dists, dense_idx = self._index.kneighbors(qd, n_neighbors=self.k)
        # Convert distance → score (cosine sim = 1-dist)
        dense_scores = np.zeros(len(self._dense))
        dense_scores[dense_idx[0]] = 1 - dense_dists[0]

        # Lexical side
        lex_scores = bm25_scores(q_text)

        # Hybrid fusion
        fused = self.alpha * lex_scores + (1 - self.alpha) * dense_scores
        ranked = np.argsort(fused)[::-1][: self.k]
        return [train_ids[i] for i in ranked]


In [18]:
# ── Cell 6: Evaluation Helpers ────────────────────────────────────────────────
def recall_at_k(retrieved: list[str], relevant: str) -> float:
    return float(relevant in retrieved)


def ndcg_at_k(retrieved: list[str], relevant: str, k: int = 5) -> float:
    if relevant not in retrieved[:k]:
        return 0.0
    rank = retrieved[:k].index(relevant) + 1
    return 1.0 / np.log2(rank + 1)


def evaluate(retriever: HybridRetriever,
             queries_text: list[str],
             queries_emb: np.ndarray,
             gold_ids: list[str]) -> dict:
    recalls, ndcgs, latencies = [], [], []
    for qt, qe, gid in zip(queries_text, queries_emb, gold_ids):
        t0 = time.perf_counter()
        retrieved = retriever.query(qt, qe)
        latencies.append(time.perf_counter() - t0)
        recalls.append(recall_at_k(retrieved, gid))
        ndcgs.append(ndcg_at_k(retrieved, gid, k=TOP_K_EVAL))
    return {
        f"Recall@{TOP_K_EVAL}": np.mean(recalls),
        f"NDCG@{TOP_K_EVAL}": np.mean(ndcgs),
        "p95_latency_ms": np.percentile(latencies, 95) * 1000,
    }


# ── Baseline (default params) ─────────────────────────────────────────────────
baseline = HybridRetriever(alpha=0.5, svd_dim=None,
                           norm="l2", metric="cosine", k=5).fit(corpus_emb)
baseline_metrics = evaluate(baseline, gold_queries, query_emb, gold_doc_ids)
print("\n Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")



 Baseline metrics:
  Recall@5: 1.0000
  NDCG@5: 0.9826
  p95_latency_ms: 2.2297


In [19]:
#  ── Cell 7: Optuna AutoML Search ──────────────────────────────────────────────
def objective(trial: optuna.Trial) -> float:
    k = trial.suggest_int("k", 1, 20)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)
    svd_dim = trial.suggest_categorical("svd_dim",
                                        [None, 64, 128, 256])
    norm = trial.suggest_categorical("norm", ["l2", "none"])
    metric = trial.suggest_categorical("metric",
                                       ["cosine", "euclidean"])

    # Build & fit retriever on train embeddings
    ret = HybridRetriever(alpha=alpha, svd_dim=svd_dim,
                          norm=norm, metric=metric, k=k)
    try:
        ret.fit(corpus_emb)
        m = evaluate(ret, gold_queries, query_emb, gold_doc_ids)
    except Exception:
        return 0.0  # penalise bad configs

    # Multi-objective scalarisation: maximise quality, penalise latency
    latency_penalty = max(0, m["p95_latency_ms"] - 500) / 1000
    score = (m[f"NDCG@{TOP_K_EVAL}"] + m[f"Recall@{TOP_K_EVAL}"]) / 2 \
            - 0.05 * latency_penalty

    # Store raw metrics as user attributes for later analysis
    trial.set_user_attr("NDCG", m[f"NDCG@{TOP_K_EVAL}"])
    trial.set_user_attr("Recall", m[f"Recall@{TOP_K_EVAL}"])
    trial.set_user_attr("p95_ms", m["p95_latency_ms"])
    return score


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42),
    study_name="knn_hybrid_automl"
)
study.optimize(objective, n_trials=N_TRIALS,
               show_progress_bar=True)

best = study.best_trial
best_params = best.params
best_ndcg = best.user_attrs["NDCG"]
best_recall = best.user_attrs["Recall"]
best_lat = best.user_attrs["p95_ms"]

print(f"\n Best params : {best_params}")
print(f"   NDCG@5      : {best_ndcg:.4f}  (baseline {baseline_metrics[f'NDCG@{TOP_K_EVAL}']:.4f})")
print(f"   Recall@5    : {best_recall:.4f}  (baseline {baseline_metrics[f'Recall@{TOP_K_EVAL}']:.4f})")
print(f"   p95 latency : {best_lat:.1f} ms")

Best trial: 14. Best value: 0.998155: 100%|██████████| 40/40 [00:11<00:00,  3.39it/s]


 Best params : {'k': 11, 'alpha': 0.11537551863842466, 'svd_dim': 256, 'norm': 'none', 'metric': 'cosine'}
   NDCG@5      : 0.9963  (baseline 0.9826)
   Recall@5    : 1.0000  (baseline 1.0000)
   p95 latency : 1.9 ms


In [ ]:
# ── Cell 8: AutoML Results Plot ───────────────────────────────────────────────
trial_df = study.trials_dataframe()

GRAPH_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "Graphs_for_prof")
os.makedirs(GRAPH_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(trial_df["value"], color="steelblue", alpha=0.6)
axes[0].axhline(trial_df["value"].cummax().iloc[-1],
                ls="--", color="red", label="best so far")
axes[0].set_title("Optuna objective over trials")
axes[0].set_xlabel("Trial");
axes[0].legend()

trial_df["user_attrs_NDCG"].plot(ax=axes[1], color="green", alpha=0.7)
axes[1].set_title("NDCG@5 per trial");
axes[1].set_xlabel("Trial")

trial_df["user_attrs_p95_ms"].plot(ax=axes[2], color="orange", alpha=0.7)
axes[2].set_title("p95 Latency (ms) per trial");
axes[2].set_xlabel("Trial")

plt.tight_layout()
plt.savefig(os.path.join(GRAPH_DIR, "automl_search.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved → Graphs_for_prof/automl_search.png")


In [21]:
# ── Cell 9: River — Adaptive Hybrid Weight from Feedback ──────────────────────

from river import linear_model, preprocessing, metrics, drift, optim

class RiverHybridAdapter:
    """
    Incrementally learns the best hybrid alpha per query from binary feedback.
    Features: BM25 top-1 score, dense top-1 score, query length.
    """
    def __init__(self):
        self.model = (
            preprocessing.StandardScaler() |
            linear_model.LogisticRegression(
                optimizer=optim.SGD(lr=0.01),
                l2=1e-4
            )
        )
        self.adwin      = drift.ADWIN(delta=0.002)
        self.metric     = metrics.Accuracy()
        self.prequential: list[dict] = []   # rolling log
        self.n_drifts   = 0

    def _features(self, q_text: str, q_emb: np.ndarray) -> dict:
        bm25_s    = bm25_scores(q_text)
        qd        = q_emb.reshape(1, -1)
        dense_s   = (corpus_emb @ qd.T).flatten()
        tokens    = q_text.split()
        return {
            "bm25_top"      : float(bm25_s.max()),
            "bm25_mean"     : float(bm25_s.mean()),         
            "bm25_nonzero"  : float((bm25_s > 0).sum()),    
            "dense_top"     : float(dense_s.max()),
            "dense_mean"    : float(dense_s.mean()),        
            "dense_std"     : float(dense_s.std()),         
            "q_len"         : len(tokens),
            "q_avg_word_len": float(np.mean([len(t) for t in tokens]) if tokens else 0),
        }

    def learn(self, q_text: str, q_emb: np.ndarray, helpful: int):
        x = self._features(q_text, q_emb)
        y_pred = self.model.predict_one(x)
        self.model.learn_one(x, helpful)

        if y_pred is not None:
            self.metric.update(helpful, y_pred)
            self.adwin.update(helpful)
            if self.adwin.drift_detected:
                self.n_drifts += 1
                print(f"⚡ ADWIN drift detected at step "
                      f"{len(self.prequential)} (total drifts: {self.n_drifts})")
            self.prequential.append({
                "step"     : len(self.prequential),
                "accuracy" : self.metric.get(),
                "drift"    : self.adwin.drift_detected,
            })

    def predict_alpha(self, q_text: str, q_emb: np.ndarray) -> float:
        x = self._features(q_text, q_emb)
        prob = self.model.predict_proba_one(x)
        if prob is None:
            return 0.5
        # prob[1] = P(helpful); map to alpha in [0.2, 0.8]
        return 0.2 + 0.6 * prob.get(1, 0.5)

# ── Simulate a feedback stream ────────────────────────────────────────────────
# Simulate 300 queries; feedback = 1 if retrieved contains gold_id, else 0.
# In production this comes from the UI /feedback endpoint.

adapter   = RiverHybridAdapter()
winner    = HybridRetriever(**{k: v for k, v in best_params.items()}).fit(corpus_emb)

n_sim = 600
sim_indices = np.tile(np.arange(GOLD_N), n_sim // GOLD_N + 1)[:n_sim]

DRIFT_POINT = n_sim // 2   

for step, idx in enumerate(sim_indices):
    qt, qe, gid = gold_queries[idx], query_emb[idx], gold_doc_ids[idx]

    # Use adapter's predicted alpha for this query
    alpha_pred = adapter.predict_alpha(qt, qe)
    winner.alpha = alpha_pred  # fine, just document it's intentional 

    retrieved = winner.query(qt, qe)
    helpful   = int(gid in retrieved)

    # ── Simulate concept drift at halfway point ───────────────────────────
    if step >= DRIFT_POINT and helpful == 1 and np.random.rand() < 0.7:
        helpful = 0   # inject noise / drift

    adapter.learn(qt, qe, helpful)

print(f"\n🌊 River stream complete — {n_sim} steps")
print(f"   Total drifts detected : {adapter.n_drifts}")

# ── Pre/post drift accuracy breakdown ────────────────────────────────────────
pq_temp = pd.DataFrame(adapter.prequential)
if len(pq_temp) > 0:
    pre_acc  = pq_temp[pq_temp["step"] <  DRIFT_POINT]["accuracy"].iloc[-1] \
               if DRIFT_POINT > 0 else float("nan")
    post_acc = pq_temp[pq_temp["step"] >= DRIFT_POINT]["accuracy"].iloc[-1] \
               if DRIFT_POINT < len(pq_temp) else float("nan")
    rel_change = (post_acc - pre_acc) / (pre_acc + 1e-9) * 100
    print(f"   Accuracy before drift : {pre_acc:.4f}")
    print(f"   Accuracy after  drift : {post_acc:.4f}")
    print(f"   Relative change       : {rel_change:+.1f}%  "
          f"({'degraded as expected — ADWIN response triggered' if rel_change < -5 else 'stable'})")


⚡ ADWIN drift detected at step 351 (total drifts: 1)

🌊 River stream complete — 600 steps
   Total drifts detected : 1
   Accuracy before drift : 0.9733
   Accuracy after  drift : 0.6400
   Relative change       : -34.2%  (degraded as expected — ADWIN response triggered)


In [ ]:
# ── Cell 10: Prequential Metrics Plot ─────────────────────────────────────────
pq = pd.DataFrame(adapter.prequential)
drift_steps = pq[pq["drift"]]["step"].tolist()

GRAPH_DIR = os.path.join(os.path.dirname(os.path.abspath(".")), "Graphs_for_prof")
os.makedirs(GRAPH_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pq["step"], pq["accuracy"], color="royalblue",
         linewidth=1.5, label="Prequential accuracy")
for i, ds in enumerate(drift_steps):
    ax.axvline(ds, color="red", lw=0.8, alpha=0.6, label="ADWIN drift" if i == 0 else ""
)
ax.set_title("River Online Learner — Prequential Accuracy (with ADWIN drift markers)")
ax.set_xlabel("Feedback step"); ax.set_ylabel("Accuracy")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(GRAPH_DIR, "prequential_chart.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Prequential chart saved → Graphs_for_prof/prequential_chart.png")

In [23]:
# ── Cell 11: Metrics Summary Table ────────────────────────────────────────────
summary = pd.DataFrame([
    {"Config": "Baseline (default)",
     f"NDCG@{TOP_K_EVAL}": round(baseline_metrics[f"NDCG@{TOP_K_EVAL}"], 4),
     f"Recall@{TOP_K_EVAL}": round(baseline_metrics[f"Recall@{TOP_K_EVAL}"], 4),
     "p95_ms": round(baseline_metrics["p95_latency_ms"], 2)},
    {"Config": "AutoML (Optuna best)",
     f"NDCG@{TOP_K_EVAL}": round(best_ndcg, 4),
     f"Recall@{TOP_K_EVAL}": round(best_recall, 4),
     "p95_ms": round(best_lat, 2)},
    {"Config": "River (online learner)",
     f"NDCG@{TOP_K_EVAL}": "-",
     f"Recall@{TOP_K_EVAL}": "-",
     "p95_ms": "-",
     "Prequential Acc": round(adapter.metric.get(), 4),
     "Drift Events": adapter.n_drifts},
])
print("\n Final Comparison Table:")
print(summary.to_string(index=False))
summary.to_csv("metrics_summary.csv", index=False)


 Final Comparison Table:
                Config  NDCG@5 Recall@5 p95_ms  Prequential Acc  Drift Events
    Baseline (default)  0.9826      1.0   2.23              NaN           NaN
  AutoML (Optuna best)  0.9963      1.0   1.89              NaN           NaN
River (online learner)       -        -      -             0.64           1.0


In [24]:
# ── Cell 12: Save YAML Run Card ───────────────────────────────────────────────
run_card = {
    "run_card": {
        "experiment": "D1-AutoML-kNN-Optuna",
        "date"      : time.strftime("%Y-%m-%d"),
        "seed"      : 42,
        "n_trials"  : N_TRIALS,
        "embed_model": EMBED_MODEL,
        "eval_k"    : TOP_K_EVAL,
    },
    "search_space": {
        "k"      : {"type": "int",   "low": 1,   "high": 20},
        "alpha"  : {"type": "float", "low": 0.0, "high": 1.0},
        "svd_dim": {"type": "categorical", "choices": [None, 64, 128, 256]},
        "norm"   : {"type": "categorical", "choices": ["l2", "none"]},
        "metric" : {"type": "categorical", "choices": ["cosine", "euclidean"]},
    },
    "winning_config": best_params,
    "metrics": {
        "baseline": {
            f"NDCG@{TOP_K_EVAL}"  : round(baseline_metrics[f"NDCG@{TOP_K_EVAL}"], 4),
            f"Recall@{TOP_K_EVAL}": round(baseline_metrics[f"Recall@{TOP_K_EVAL}"], 4),
            "p95_latency_ms"      : round(baseline_metrics["p95_latency_ms"], 2),
        },
        "automl_best": {
            f"NDCG@{TOP_K_EVAL}"  : round(best_ndcg, 4),
            f"Recall@{TOP_K_EVAL}": round(best_recall, 4),
            "p95_latency_ms"      : round(best_lat, 2),
        },
    },
    "online_learning": {
        "library"   : "River",
        "model"     : "LogisticRegression (SGD, l2=1e-4)",
        "drift"     : "ADWIN (delta=0.002)",
        "steps"     : n_sim,
        "n_drifts"  : adapter.n_drifts,
        "final_accuracy": round(adapter.metric.get(), 4),
    },
}

with open("run_card.yaml", "w") as f:
    yaml.dump(run_card, f, default_flow_style=False, sort_keys=False)

print("\n✅ run_card.yaml saved")
print(yaml.dump(run_card, default_flow_style=False))


✅ run_card.yaml saved
metrics:
  automl_best:
    NDCG@5: !!python/object/apply:numpy.core.multiarray.scalar
    - &id001 !!python/object/apply:numpy.dtype
      args:
      - f8
      - false
      - true
      state: !!python/tuple
      - 3
      - <
      - null
      - null
      - null
      - -1
      - -1
      - 0
    - !!binary |
      UiegibDh7z8=
    Recall@5: !!python/object/apply:numpy.core.multiarray.scalar
    - *id001
    - !!binary |
      AAAAAAAA8D8=
    p95_latency_ms: !!python/object/apply:numpy.core.multiarray.scalar
    - *id001
    - !!binary |
      PQrXo3A9/j8=
  baseline:
    NDCG@5: !!python/object/apply:numpy.core.multiarray.scalar
    - *id001
    - !!binary |
      U5YhjnVx7z8=
    Recall@5: !!python/object/apply:numpy.core.multiarray.scalar
    - *id001
    - !!binary |
      AAAAAAAA8D8=
    p95_latency_ms: !!python/object/apply:numpy.core.multiarray.scalar
    - *id001
    - !!binary |
      16NwPQrXAUA=
online_learning:
  drift: ADWIN (delta=0.002)
